# Ariel 2025 — Notebook 2: ML Training (physics-based tabular models)

Train pipeline cho nhóm **machine learning trên feature vật lý** (không phải deep sequence):
calibrate → feature engineering → Target PCA → hồi quy theo họ mô hình → **PHC sigma calibration** → chọn model theo **Ariel GLL** → lưu weights (kèm calibrated sigma) → submission.

> Chạy bằng **CPU** (sklearn). GPU chỉ giúp LightGBM/XGBoost — bật `USE_GPU` nếu cần. Build feature là bước chậm nhất; có cache để chạy lại nhanh.

In [ ]:
# === Setup: clone running branch and make modules importable ===
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Jun1801/ML_IT3190E_Project.git"
CLONE_DIR = Path("/kaggle/working/ML_IT3190E_Project")
if not CLONE_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", "running", "--single-branch", REPO_URL, str(CLONE_DIR)],
        check=True,
    )
    print("Cloned branch 'running' →", CLONE_DIR)
else:
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull"], check=True)
    print("Pulled latest →", CLONE_DIR)

CANDIDATE_SRC = [
    str(CLONE_DIR / "src"),
    "/kaggle/input/ariel-ml-src/src",
    "src", "../src",
]
for _p in CANDIDATE_SRC:
    if Path(_p).exists():
        sys.path.insert(0, _p); print("Using src from:", _p); break
else:
    print("WARNING: src not found.")

# Optional GBM deps (LightGBM/XGBoost preinstalled on Kaggle; ngboost is not):
# !pip install -q ngboost

DATA_ROOT = Path("/kaggle/input/ariel-data-challenge-2025")
OUTPUT_DIR = Path("/kaggle/working"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR = OUTPUT_DIR / "weights"; WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT exists:", DATA_ROOT.exists())


## 1. Cấu hình

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import ModelConfig
from dataset_builder import align_features_and_targets
from benchmark import benchmark_models
from training import (
    cross_validate_model, hyperparameter_search, train_model, refit_full_model,
    build_gll_weighted_ensemble,
)
from estimators import ModelFactory

# Phải KHỚP với prepare_features.ipynb để load đúng file precomputed
LIMIT = None        # None = full (khuyến nghị); đặt số nhỏ (vd 100) chỉ để smoke
TIME_BINS = 128

# ---- training knobs ----
N_COMPONENTS = 24
N_SPLITS = 5
SIGMA_CAL_FRACTION = 0.2     # giữ 20% train fold cho sigma calibration -> tránh circular NLL
USE_GPU = False              # chỉ tăng tốc lightgbm/xgboost
RANDOM_STATE = 42
MAX_PLANETS_BENCH = None     # nếu thiếu RAM ở benchmark, đặt 300-500

print("config ok")


## 2. Load precomputed features
Features được build ở **`prepare_features.ipynb`** (CPU, chạy một lần) và push lên branch `running`. Notebook này chỉ load lại — `LIMIT`/`TIME_BINS` phải khớp.

In [ ]:
PRECOMPUTED_DIR = CLONE_DIR / "precomputed"
train_csv = PRECOMPUTED_DIR / f"features_train_L{LIMIT}_T{TIME_BINS}.csv"
test_csv = PRECOMPUTED_DIR / f"features_test_T{TIME_BINS}.csv"

def _resolve(path):
    for cand in [path, OUTPUT_DIR / path.name, Path("/kaggle/input/ariel-features") / path.name]:
        if Path(cand).exists():
            return Path(cand)
    return path

train_csv, test_csv = _resolve(train_csv), _resolve(test_csv)
assert train_csv.exists(), (
    f"Không thấy {train_csv.name}. Chạy prepare_features.ipynb (cùng LIMIT/TIME_BINS) rồi push lên branch 'running'."
)
print("Train features:", train_csv, "| test:", test_csv, "(", test_csv.exists(), ")")


In [ ]:
features = pd.read_csv(train_csv)
targets = pd.read_csv(DATA_ROOT / "train.csv")
X, y, groups, target_columns = align_features_and_targets(features, targets)
Xv = X.to_numpy(dtype=float)
print("X:", Xv.shape, "| y:", y.shape, "| planets:", len(np.unique(groups)),
      "| features:", Xv.shape[1], "| targets:", y.shape[1])


## 2b. Chẩn đoán: mean có học được không?
GLL chỉ cao khi **mean dự đoán hơn mean toàn cục**. Tính R² theo từng wavelength trên một split: R²>0 nghĩa là model giải thích được biến thiên của planet đó. Nếu R²≈0 ⇒ nút thắt là feature/signal (hoặc thiếu dữ liệu), không phải tầng calibration.

In [ ]:
from training import train_model

def per_wavelength_r2(name, n_components=N_COMPONENTS):
    res = train_model(Xv, y, model_name=name,
                      model_config=ModelConfig(n_components=n_components, random_state=RANDOM_STATE),
                      validation_fraction=0.25, groups=groups, random_state=RANDOM_STATE)
    yv = y[res.validation_index]; pv = res.prediction.mu
    ss_res = ((yv - pv) ** 2).sum(axis=0)
    ss_tot = ((yv - yv.mean(axis=0)) ** 2).sum(axis=0)
    return 1.0 - ss_res / np.maximum(ss_tot, 1e-12)

r2s = {}
for name in ["ridge", "extra_trees"]:
    r2 = per_wavelength_r2(name); r2s[name] = r2
    print(f"{name:12s} mean R²={r2.mean():+.3f} | median={np.median(r2):+.3f} | "
          f"% wavelength R²>0: {(r2 > 0).mean() * 100:.0f}%")

plt.figure(figsize=(9, 3))
for name, r2 in r2s.items():
    plt.plot(r2, label=name, lw=1)
plt.axhline(0, color="k", lw=0.6); plt.ylim(-1, 1)
plt.xlabel("wavelength index"); plt.ylabel("R² (val)"); plt.legend(); plt.title("Mean predictiveness per wavelength")
plt.tight_layout(); plt.show()


## 3. So sánh các họ mô hình (benchmark, official GLL)
Cùng GroupKFold folds, cùng `n_components`. Model thiếu optional dep → `skipped`.

In [ ]:
MODEL_NAMES = [
    "ridge", "lasso", "elastic_net",
    "bayesian_ridge", "ard",
    "svr", "kernel_ridge", "knn",
    "random_forest", "extra_trees", "boosting", "hist_gradient_boosting",
    "mlp", "lightgbm", "xgboost",
]
# gaussian_process / ngboost rất chậm trên nhiều planet -> thêm nếu cần.

Xb, yb, gb = Xv, y, groups
if MAX_PLANETS_BENCH is not None:
    keep = np.isin(groups, np.unique(groups)[:MAX_PLANETS_BENCH])
    Xb, yb, gb = Xv[keep], y[keep], groups[keep]
    print("Benchmark subset:", Xb.shape)

bench_cfg = ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE,
                        sigma_per_target=True, use_gpu=USE_GPU)  # PHC per-wavelength: so sánh GLL công bằng
result = benchmark_models(Xb, yb, model_names=MODEL_NAMES, model_config=bench_cfg,
                          n_splits=N_SPLITS, groups=gb, random_state=RANDOM_STATE,
                          sigma_cal_fraction=SIGMA_CAL_FRACTION)
table = result.to_frame()
table.to_csv(OUTPUT_DIR / "benchmark.csv", index=False)
best_bench = result.best("ariel_gll_score", maximize=True)
print("Best by Ariel GLL:", best_bench.model_name, "=", round(best_bench.metrics["ariel_gll_score"], 4))
table


In [ ]:
ok = table[table["status"] == "ok"].sort_values("ariel_gll_score")
plt.figure(figsize=(8, max(3, 0.4 * len(ok))))
plt.barh(ok["model"], ok["ariel_gll_score"], color="steelblue")
plt.xlabel("Ariel GLL score (higher = better)"); plt.title("Model family comparison")
plt.tight_layout(); plt.show()


## 4. Hyperparameter search (chọn theo Ariel GLL, có PHC)
Quét model × `n_components`, calibrate sigma per-wavelength (PHC), chọn best theo `ariel_gll_score`.

In [ ]:
search = hyperparameter_search(
    Xv, y,
    model_names=["bayesian_ridge", "ridge", "random_forest", "kernel_ridge"],
    n_components_grid=[16, 24, 32],
    base_config=ModelConfig(random_state=RANDOM_STATE, sigma_per_target=True, use_gpu=USE_GPU),
    n_splits=N_SPLITS, groups=groups, random_state=RANDOM_STATE,
    selection_metric="ariel_gll_score", sigma_cal_fraction=SIGMA_CAL_FRACTION,
)
bestc = search.best_candidate
print("Best:", bestc.model_name, "| n_components =", bestc.model_config.n_components,
      "| GLL =", round(bestc.mean_metrics["ariel_gll_score"], 4))


## 5. Train + lưu weights (kèm calibrated sigma)
Mỗi model fit bằng `train_model` trên train split và **calibrate σ trên holdout** (PHC per-wavelength) → artifact `.joblib` có σ đã hiệu chỉnh, dùng predict ngay.

In [ ]:
import joblib, gc

SAVE_MODELS = sorted(set([bestc.model_name, "bayesian_ridge", "ridge", "random_forest"]))
save_cfg = ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE,
                       sigma_per_target=True, use_gpu=USE_GPU)
VAL_FRACTION = 0.2

saved = []
for name in SAVE_MODELS:
    cfg = bestc.model_config if name == bestc.model_name else save_cfg
    try:
        res = train_model(Xv, y, model_name=name, model_config=cfg,
                          validation_fraction=VAL_FRACTION, groups=groups, random_state=RANDOM_STATE)
        artifact = {
            "model": res.model,
            "feature_columns": list(X.columns),
            "target_columns": target_columns,
            "model_name": name,
            "model_config": cfg,
            "val_metrics": res.evaluation.as_dict(),
        }
        path = WEIGHTS_DIR / f"{name}.joblib"
        joblib.dump(artifact, path)
        saved.append(name)
        print(f"saved: {path}  (val GLL={res.evaluation.ariel_gll_score:.4f})")
    except Exception as exc:
        print("skip", name, "->", type(exc).__name__, exc)
    finally:
        gc.collect()

# GLL-weighted family mixture (calibrated members)
ens = build_gll_weighted_ensemble(
    Xv, y, model_names=("bayesian_ridge", "random_forest", "kernel_ridge"),
    model_config=ModelConfig(n_components=N_COMPONENTS, random_state=RANDOM_STATE, sigma_per_target=True, use_gpu=USE_GPU),
    validation_fraction=VAL_FRACTION, groups=groups, random_state=RANDOM_STATE,
)
joblib.dump({"ensemble": ens.ensemble, "model_names": ens.model_names, "weights": ens.weights,
             "feature_columns": list(X.columns), "target_columns": target_columns},
            WEIGHTS_DIR / "gll_weighted_ensemble.joblib")
print("ensemble weights:", dict(zip(ens.model_names, np.round(ens.weights, 3))))
print("Saved", len(saved), "models + ensemble to", WEIGHTS_DIR)


## 6. Submission
Dùng best model (đã calibrate) để predict trên test features.

In [ ]:
from submission import infer_submission_schema, predict_submission, save_submission

if test_csv.exists():
    art = joblib.load(WEIGHTS_DIR / f"{bestc.model_name}.joblib")
    test_features = pd.read_csv(test_csv)
    for col in art["feature_columns"]:
        if col not in test_features.columns:
            test_features[col] = 0.0
    sample = pd.read_csv(DATA_ROOT / "sample_submission.csv")
    schema = infer_submission_schema(sample_submission=sample)
    submission = predict_submission(art["model"], test_features,
                                    feature_columns=art["feature_columns"], schema=schema)
    save_submission(submission, OUTPUT_DIR / "submission.csv")
    print("Saved submission:", submission.shape, "->", OUTPUT_DIR / "submission.csv")
    submission.head()
else:
    print("No test features. Set BUILD_TEST=True and rebuild.")


## Tổng kết
Đã lưu: `benchmark.csv`, weights model (`weights/*.joblib`, kèm calibrated σ + val_metrics), `gll_weighted_ensemble.joblib`, và `submission.csv`.
Để chạy nhóm **deep sequence**, dùng `prepare_sequence.ipynb` (CPU) + `run_deep_learning.ipynb` (GPU).